In [0]:
#Reads the credentials from the Databricks Secret Scope
import os
from google.cloud import bigquery
from google.oauth2 import service_account

gcp_key_json = dbutils.secrets.get(scope="gcp-bigquery", key="service-account-key")

with open("/tmp/gcp_key.json", "w") as f:
    f.write(gcp_key_json)

#Stores the credentials for later use
credentials = service_account.Credentials.from_service_account_file("/tmp/gcp_key.json")
client = bigquery.Client(credentials=credentials, project=credentials.project_id)

In [0]:
#Creates the schema
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

#Defines the dataset of the tables
DATASET = "basedosdados.br_inep_avaliacao_alfabetizacao"

#List of tables to ingest
tables = {
    "uf":                            f"{DATASET}.uf",
    "municipio":                     f"{DATASET}.municipio",
    "dicionario":                    f"{DATASET}.dicionario",
    "meta_alfabetizacao_brasil":     f"{DATASET}.meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf":         f"{DATASET}.meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio":  f"{DATASET}.meta_alfabetizacao_municipio"
}

#Queries each table and inserts them into the Bronze schema
for table_name, table_path in tables.items():
    query = f"SELECT * FROM `{table_path}`"
    df_pd = client.query(query).to_dataframe()

    df_spark = spark.createDataFrame(df_pd)
    df_spark.write.format("delta").mode("overwrite") \
        .saveAsTable(f"bronze.{table_name}")

    print(f"{table_name} written with {len(df_pd)} lines ingested")

In [0]:
#Batch ingestion for students
DATASET = "basedosdados.br_inep_avaliacao_alfabetizacao"

#List of tables to ingest
tables = {
    "alunos":                        f"{DATASET}.alunos"
}

#Queries each table and inserts them into the Bronze schema
for table_name, table_path in tables.items():
    query = f"SELECT * FROM `{table_path}`"
    df_pd = client.query(query).to_dataframe()

    df_spark = spark.createDataFrame(df_pd)
    df_spark.write.format("delta").mode("overwrite") \
        .saveAsTable(f"bronze.{table_name}")

    print(f"{table_name} written with {len(df_pd)} lines ingested")